In [1]:
import pandas

weather_df   = pandas.read_csv("../../data/raw/hanwoo_weather.csv")
new_weather1 = pandas.read_csv("../../data/raw/OBS_ASOS_DD_20260606112118.csv", encoding="cp949")
new_weather2 = pandas.read_csv("../../data/raw/OBS_ASOS_DD_20260606114708.csv", encoding="cp949")
new_weather3 = pandas.read_csv("../../data/raw/OBS_ASOS_DD_20260606144527.csv", encoding="cp949")

# =====================================================
# 날짜 범위 확인
# =====================================================
print("=== 날짜 범위 확인 ===")
print(f"new_weather1: {new_weather1['Date'].min()} ~ {new_weather1['Date'].max()}")
print(f"new_weather2: {new_weather2['Date'].min()} ~ {new_weather2['Date'].max()}")
print(f"new_weather3: {new_weather3['Date'].min()} ~ {new_weather3['Date'].max()}")
print(f"weather_df:   {weather_df['date'].min()} ~ {weather_df['date'].max()}")

# =====================================================
# weather_df 관측소 기준으로 필터링
# =====================================================
stn_list = set(weather_df["stn"].unique())
print(f"\n=== 관측소 필터링 ===")
print(f"weather_df 관측소 수: {len(stn_list)}")

new_weather1_filtered = new_weather1[new_weather1["Stn"].isin(stn_list)].copy()
new_weather2_filtered = new_weather2[new_weather2["Stn"].isin(stn_list)].copy()
new_weather3_filtered = new_weather3[new_weather3["Stn"].isin(stn_list)].copy()

print(f"new_weather1: {len(new_weather1):,} → {len(new_weather1_filtered):,}")
print(f"new_weather2: {len(new_weather2):,} → {len(new_weather2_filtered):,}")
print(f"new_weather3: {len(new_weather3):,} → {len(new_weather3_filtered):,}")

# =====================================================
# 컬럼명 통일
# =====================================================
rename_cols = {
    "Stn": "stn", "Date": "date",
    "Ta_min": "ta_min", "Ta_max": "ta_max",
    "Rn_day": "rn_day", "Rhm_avg": "rhm_avg", "Ws_davg": "ws_davg"
}
new_weather1_filtered = new_weather1_filtered.rename(columns=rename_cols)
new_weather2_filtered = new_weather2_filtered.rename(columns=rename_cols)
new_weather3_filtered = new_weather3_filtered.rename(columns=rename_cols)

# =====================================================
# 4개 파일 합치기
# =====================================================
weather_cols = ["stn", "date", "ta_min", "ta_max", "rn_day", "rhm_avg", "ws_davg"]
weather_all = pandas.concat([
    new_weather3_filtered[weather_cols],
    new_weather1_filtered[weather_cols],
    new_weather2_filtered[weather_cols],
    weather_df[weather_cols]
], ignore_index=True)

# 날짜 변환 및 정렬
weather_all["date"] = pandas.to_datetime(weather_all["date"])
weather_all = weather_all.sort_values(["stn", "date"]).reset_index(drop=True)

# =====================================================
# 합친 결과 확인
# =====================================================
print("\n=== 합친 결과 ===")
print(f"행 수: {len(weather_all):,}")
print(f"날짜 범위: {weather_all['date'].min()} ~ {weather_all['date'].max()}")
print(f"관측소 수: {weather_all['stn'].nunique()}")

# 중복 확인
duplicates = weather_all.duplicated(["stn", "date"]).sum()
print(f"stn+date 중복: {duplicates:,}")

# 결측 확인
print("\n결측치 현황:")
print(weather_all[weather_cols[2:]].isnull().sum())

=== 날짜 범위 확인 ===
new_weather1: 2004-01-01 ~ 2014-01-01
new_weather2: 2014-01-02 ~ 2019-12-31
new_weather3: 1996-01-01 ~ 2003-12-31
weather_df:   2020-01-01 ~ 2025-12-31

=== 관측소 필터링 ===
weather_df 관측소 수: 444
new_weather1: 306,872 → 281,292
new_weather2: 206,831 → 193,312
new_weather3: 220,244 → 197,385

=== 합친 결과 ===
행 수: 1,645,237
날짜 범위: 1996-01-01 00:00:00 ~ 2025-12-31 00:00:00
관측소 수: 444
stn+date 중복: 0

결측치 현황:
ta_min         36
ta_max         35
rn_day     425823
rhm_avg       718
ws_davg       452
dtype: int64


In [2]:
import os
os.makedirs("../../data/processed", exist_ok=True)
weather_all.to_csv("../data/processed/step_4-1_weather.csv", index=False, encoding="utf-8-sig")
print(f"저장 완료: {weather_all.shape}")

저장 완료: (1645237, 7)
